## Data Processing to Discretize numerical features for Association Rules Mining

In [1]:
import pandas as pd

load_df = pd.read_csv("data/force/f_train.csv")
drop_columns = ['RSHA', 'RMED', 'GR', 'PEF', 'SP', 'ROP', 'DRHO', 'RDEP', 'WELL']
load_df = load_df.drop(drop_columns, axis=1)
load_df = load_df.dropna()
df = load_df.copy()

# Categorize Lithology
litho_code = {30000: "Sandstone", 65030: "SandstoneShale", 65000: "Shale", 80000: "Marl", 
            74000: "Dolomite", 70000: "Limestone", 70032: "Chalk", 88000: "Halite", 
            86000: "Anhydrite", 99000: "Tuff", 90000: "Coal", 93000: "Basement"}
df["LITHOLOGY"] = df["LITHOLOGY"].map(litho_code)

# Categorize Borehole Size
BS_code = {12.250000954: 'BS122', 8.500000: 'BS85', 17.500000: 'BS175', 14.750000: 'BS147', 13.009029388: 'BS130', 9.7999038696: 'BS97', 
           9.5535526276: 'BS95', 11.760092735: 'BS117', 17.317329407: 'BS173', 9.4227342606: 'BS94'}
df["BS"] = df["BS"].map(BS_code)

# Re-format values of GROUP
df.GROUP = df.GROUP.str.replace(" GP.", "")

# Re-format values of FORMATION
df.FORMATION = df.FORMATION.str.replace(" ", "")
df.FORMATION = df.FORMATION.str.replace(".", "")

# Categories of DEPTH_MD
depth_categories = {
    "ShallowDepth": (0, 500),
    "InterDepth": (500, 1500),
    "DeepDepth": (1500, 3000),
    "UltraDDepth": (3000, 4000)}
def categorize_depth(depth):
    for category, (lower, upper) in depth_categories.items():
        if lower <= depth < upper:
            return category
    return "Unknown"
df["DEPTH_MD"] = df["DEPTH_MD"].apply(categorize_depth)

# Categories of CALI
caliper_categories = {
    "VNarrowCaliper": (5, 10),
    "NarrowCaliper": (10, 15),
    "NormalCaliper": (15, 20),
    "EnlargedCaliper": (20, 30),
}
def categorize_caliper(caliper):
    for category, (lower, upper) in caliper_categories.items():
        if lower <= caliper < upper:
            return category
    return "Unknown"
df["CALI"] = df["CALI"].apply(categorize_caliper)

# Categories of RHOB
rhob_categories = {
    "LowRHOB": (1, 2.0),
    "MidRHOB ": (2.0, 2.5),
    "HighRHOB": (2.5, 3.0)}
def categorize_rhob(rhob):
    for category, (lower, upper) in rhob_categories.items():
        if lower <= rhob < upper:
            return category
    return "Unknown"
df["RHOB"] = df["RHOB"].apply(categorize_rhob)

# Categories of DTC
dtc_categories = {
    "LowVelocity": (0, 75),
    "MidVelocity ": (75, 140),
    "HighVelocity": (140, 220)}
def categorize_dtc(DTC):
    for category, (lower, upper) in dtc_categories.items():
        if lower <= DTC < upper:
            return category
    return "Unknown"
df["DTC"] = df["DTC"].apply(categorize_dtc)

# Categories of NPHI
nphi_categories = {
    "LowPorosity": (0, .3),
    "MidPorosity ": (.3, .6),
    "HighPorosity": (.6, 1.1)}
def categorize_nphi(nphi):
    for category, (lower, upper) in nphi_categories.items():
        if lower <= nphi < upper:
            return category
    return "Unknown"
df["NPHI"] = df["NPHI"].apply(categorize_nphi)

#### Before Discretizing

In [2]:
load_df.head()

,DEPTH_MD,GROUP,FORMATION,CALI,RHOB,NPHI,DTC,BS,LITHOLOGY
0,1149.648,HORDALAND GP.,Utsira Fm.,17.482092,2.063168,0.541850,134.226379,17.5,65000
1,1149.800,HORDALAND GP.,Utsira Fm.,17.395611,2.051136,0.545401,134.824799,17.5,65000
2,1149.952,HORDALAND GP.,Utsira Fm.,17.364607,2.041540,0.548953,135.037079,17.5,65000
3,1150.104,HORDALAND GP.,Utsira Fm.,17.371887,2.035698,0.549356,134.500336,17.5,65000
4,1150.256,HORDALAND GP.,Utsira Fm.,17.370705,2.029099,0.543351,132.162399,17.5,65000


In [3]:
load_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42181 entries, 0 to 42180
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   DEPTH_MD   42181 non-null  float64
 1   GROUP      42181 non-null  object 
 2   FORMATION  42181 non-null  object 
 3   CALI       42181 non-null  float64
 4   RHOB       42181 non-null  float64
 5   NPHI       42181 non-null  float64
 6   DTC        42181 non-null  float64
 7   BS         42181 non-null  float64
 8   LITHOLOGY  42181 non-null  int64  
dtypes: float64(6), int64(1), object(2)
memory usage: 2.9+ MB


#### After Discretizing

In [4]:
df = df[df.LITHOLOGY != "Shale"]
df = df.reset_index(drop=True)

df.drop(columns=['LITHOLOGY'], inplace=True)

In [5]:
df.head()

,DEPTH_MD,GROUP,FORMATION,CALI,RHOB,NPHI,DTC,BS
0,InterDepth,HORDALAND,UtsiraFm,NormalCaliper,MidRHOB,MidPorosity,MidVelocity,BS175
1,InterDepth,HORDALAND,UtsiraFm,NormalCaliper,MidRHOB,MidPorosity,MidVelocity,BS175
2,InterDepth,HORDALAND,UtsiraFm,NormalCaliper,MidRHOB,HighPorosity,MidVelocity,BS175
3,InterDepth,HORDALAND,UtsiraFm,NormalCaliper,MidRHOB,HighPorosity,MidVelocity,BS175
4,InterDepth,HORDALAND,UtsiraFm,NormalCaliper,MidRHOB,HighPorosity,MidVelocity,BS175


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13972 entries, 0 to 13971
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   DEPTH_MD   13972 non-null  object
 1   GROUP      13972 non-null  object
 2   FORMATION  13972 non-null  object
 3   CALI       13972 non-null  object
 4   RHOB       13972 non-null  object
 5   NPHI       13972 non-null  object
 6   DTC        13972 non-null  object
 7   BS         13972 non-null  object
dtypes: object(8)
memory usage: 873.4+ KB


#### Save to files

In [7]:
df_noShale =df.reset_index(drop=True)
df_noShale['items'] = df_noShale['BS'].astype(str) +', '+ df_noShale['GROUP'].astype(str) +', '+ df_noShale['FORMATION'].astype(str) +', '+ df_noShale['DEPTH_MD'].astype(str) +', '+ df_noShale['CALI'].astype(str) +', '+ df_noShale['RHOB'].astype(str) +', '+ df_noShale['DTC'].astype(str) +', '+ df_noShale['NPHI'].astype(str)
df_noShale['items'][:5]
# Save items as transaction data to txt file, include the index
df_noShale['items'].to_csv('data/arm/transaction_data.txt', header=False, index=False)

In [8]:
# df.LITHOLOGY without Shale
df_noShale.to_csv("data/arm/arm_data_noShale.csv", index=True)